# 🛫 Kenya Airways Water Bottling Plant — ML Intelligence Simulation
### Final Year Capstone Project (ICS 4D) — Strathmore University
**Student:** Oyieri Jermaine Obed (166588)  
**Supervisor:** Julliet Kirui  

---
### Overview
This notebook simulates and demonstrates the **Machine Learning Intelligence Layer** of the **KQ Water Sales & Order Intelligence System**.
The three models demonstrated here are:
1. **SARIMAX Time-Series Demand Forecasting:** Predicting future bulk water demand (litres/cases) for production and inventory planning.
2. **RFM-Based K-Means Customer Segmentation:** Clustering internal departments and corporate distributors based on Recency, Frequency, and Monetary value.
3. **Isolation Forest Order Anomaly Detection:** Flagging unusual order volumes and sudden transaction surges in real time.

## 1. Setup & Environment Dependencies

In [ ]:
# In Google Colab, statsmodels, scikit-learn, seaborn, and pandas are pre-installed
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# Stats & ML Libraries
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
print("Libraries successfully imported!")

## 2. Dataset Ingestion: Kenya Airways Water Operations
Loading 12 months of historical B2B transaction records extracted from the system's SQLite database.
Includes prices in Kenyan Shillings (KES), customer segments (Internal Departments, Catering, Distributors), and real operational volumes.

In [ ]:
# Load the dataset directly from the project's GitHub repository
dataset_url = "https://raw.githubusercontent.com/aawbed/kq-sales-order-intelligence/main/kq_historical_sales_2025_2026.csv"

df_orders = pd.read_csv(dataset_url)

# Convert Date column to proper datetime objects
df_orders['date'] = pd.to_datetime(df_orders['Date'])

# Rename columns to match our ML pipeline variables
df_orders = df_orders.rename(columns={
    'Order ID': 'order_id',
    'Customer Name': 'customer_name',
    'Account Type': 'account_type',
    'Product Name': 'product_name',
    'Quantity': 'quantity',
    'Unit Price (KSh)': 'unit_price_kes',
    'Total Amount (KSh)': 'total_amount_kes',
    'Total Litres': 'total_litres'
})

print("Dataset successfully loaded!\n")
print(f"Total Order Lines: {len(df_orders):,}")
print(f"Total Revenue Analyzed: KSh {df_orders['total_amount_kes'].sum():,.2f}\n")

df_orders.head()

## 3. Exploratory Data Analysis (EDA)
Let's visualize the operational order patterns to observe seasonal trends and product demand distributions.

In [ ]:
# Monthly sales aggregation
df_orders['month_year'] = pd.to_datetime(df_orders['date']).dt.to_period('M')
monthly_summary = df_orders.groupby('month_year')['total_amount_kes'].sum() / 1e6

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Subplot 1: Monthly revenue in Millions KES
monthly_summary.plot(kind='bar', color='#c8102e', ax=ax1)
ax1.set_title("Monthly Water Sales Revenue (Millions KES)", fontsize=13, fontweight='bold')
ax1.set_ylabel("Revenue (M KSh)")
ax1.set_xlabel("Billing Month")
ax1.tick_params(axis='x', rotation=45)

# Subplot 2: Order distribution by product
product_counts = df_orders['product_name'].value_counts()
product_counts.plot(kind='barh', color='#0b2240', ax=ax2)
ax2.set_title("Sales Volume by Product Line Item", fontsize=13, fontweight='bold')
ax2.set_xlabel("Number of Orders Placed")

plt.tight_layout()
plt.show()

--- 
## 🧠 Model 1: SARIMAX Demand Forecasting
**Objective:** Predict the daily water volume (litres) required by flight catering and corporate buyers for the next 30 days.
**Algorithm:** Seasonal Auto-Regressive Integrated Moving Average with eXogenous factors ($p=1, d=1, q=1, P=1, D=1, Q=0, s=7$ for weekly flight schedules).

In [ ]:
# Prepare daily time-series
df_daily = df_orders.groupby('date')['total_litres'].sum().reset_index()
df_daily['date'] = pd.to_datetime(df_daily['date'])
df_daily = df_daily.set_index('date').asfreq('D', fill_value=0)

# Train/Validation Split (Last 30 days as validation)
train_data = df_daily[:-30]
val_data = df_daily[-30:]

# Fit SARIMAX Model with 7-day seasonality (airline catering cycle)
sarimax_model = SARIMAX(
    train_data['total_litres'],
    order=(1, 1, 1),
    seasonal_order=(1, 1, 0, 7),
    enforce_stationarity=False,
    enforce_invertibility=False
)
model_fit = sarimax_model.fit(disp=False)

# Predict validation period and future 30 days
val_pred = model_fit.get_forecast(steps=30)
pred_mean = val_pred.predicted_mean
conf_int = val_pred.conf_int()

# Model Evaluation Metrics
mae = mean_absolute_error(val_data['total_litres'], pred_mean)
rmse = np.sqrt(mean_squared_error(val_data['total_litres'], pred_mean))
print("=== SARIMAX Model Evaluation ===")
print(f"Validation Mean Absolute Error (MAE): {mae:.2f} Litres")
print(f"Validation Root Mean Squared Error (RMSE): {rmse:.2f} Litres")

# Visualization
plt.figure(figsize=(15, 6))
plt.plot(train_data.index[-60:], train_data['total_litres'][-60:], label='Historical Actuals (Past 60 Days)', color='#0b2240', lw=2)
plt.plot(val_data.index, val_data['total_litres'], label='Actual Test Period', color='#2b8a3e', lw=2, linestyle='--')
plt.plot(val_data.index, pred_mean, label='SARIMAX 30-Day Forecast', color='#c8102e', lw=2.5)
plt.fill_between(val_data.index, np.maximum(0, conf_int.iloc[:, 0]), conf_int.iloc[:, 1], color='#c8102e', alpha=0.15, label='95% Confidence Interval')

plt.title("KQ Bottling Plant — 30-Day Demand Forecasting (Litres)", fontsize=14, fontweight='bold')
plt.xlabel("Timeline (Date)")
plt.ylabel("Water Volume (Litres)")
plt.legend(loc='upper left', frameon=True)
plt.tight_layout()
plt.show()

--- 
## 👥 Model 2: Customer Segmentation (RFM + K-Means)
**Objective:** Group client accounts based on their purchase dynamics to enable tailored sales follow-ups.
* **R (Recency):** Number of days since client's last order.
* **F (Frequency):** Total number of bookings/orders over the year.
* **M (Monetary):** Total expenditure in KES.

In [ ]:
# Calculate RFM per customer
snapshot_date = df_orders['date'].max() + timedelta(days=1)

rfm = df_orders.groupby('customer_name').agg({
    'date': lambda x: (snapshot_date - x.max()).days,
    'order_id': 'count',
    'total_amount_kes': 'sum'
}).reset_index()

rfm.columns = ['Customer', 'Recency', 'Frequency', 'Monetary']

# Feature scaling
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['Recency', 'Frequency', 'Monetary']])

# Optimal K selection & clustering
k = 3
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

# Label clusters meaningfully by sorting on Monetary Value
cluster_order = rfm.groupby('Cluster')['Monetary'].mean().sort_values(ascending=False).index
labels_map = {
    cluster_order[0]: 'VIP / High-Volume',
    cluster_order[1]: 'Consistent Regulars',
    cluster_order[2]: 'At-Risk / Dormant'
}
rfm['Segment'] = rfm['Cluster'].map(labels_map)

# Visualizing the clusters
plt.figure(figsize=(12, 6))
palette = {'VIP / High-Volume': '#2b8a3e', 'Consistent Regulars': '#1c7ed6', 'At-Risk / Dormant': '#e03131'}
sns.scatterplot(
    data=rfm,
    x='Frequency',
    y='Monetary',
    hue='Segment',
    palette=palette,
    s=180,
    style='Segment',
    alpha=0.9
)

for i, row in rfm.iterrows():
    plt.text(row['Frequency'] + 0.3, row['Monetary'], row['Customer'], fontsize=8.5, alpha=0.85)

plt.title("Customer Segmentation: Frequency vs. Total Spend (KES)", fontsize=13, fontweight='bold')
plt.xlabel("Order Frequency (Annual Total Orders)")
plt.ylabel("Total Monetary Value (KSh)")
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()

# Display segment profiles summary
print("=== Cluster Summary Statistics ===")
display(rfm.groupby('Segment')[['Recency', 'Frequency', 'Monetary']].mean().round(2))

--- 
## 🚨 Model 3: Order Anomaly Detection (Isolation Forest)
**Objective:** Intercept and flag unusual orders in real time before fulfillment (e.g. erratic spikes, suspected data-entry errors, or supply chain shocks).
**Algorithm:** `IsolationForest` (Contamination = 5%).

In [ ]:
# Feature matrix for anomaly detection
# 1. Quantity ordered
# 2. Order value (KSh)
# 3. Frequency profile of placing customer
cust_freq = df_orders.groupby('customer_name')['order_id'].transform('count')
df_orders['customer_order_freq'] = cust_freq

features = df_orders[['quantity', 'total_amount_kes', 'customer_order_freq']]

# Train Isolation Forest
iso_forest = IsolationForest(contamination=0.04, random_state=42)
df_orders['anomaly_pred'] = iso_forest.fit_predict(features)
df_orders['anomaly_score'] = iso_forest.decision_function(features)

# -1 represents anomaly, 1 represents normal
df_orders['is_flagged'] = df_orders['anomaly_pred'] == -1

flagged_orders = df_orders[df_orders['is_flagged']]
print(f"Total Orders Evaluated: {len(df_orders)}")
print(f"Flagged Irregular/Anomalous Orders: {len(flagged_orders)}")

# Visualization
plt.figure(figsize=(14, 6))
plt.scatter(
    df_orders[~df_orders['is_flagged']]['quantity'],
    df_orders[~df_orders['is_flagged']]['total_amount_kes'],
    c='#1c7ed6', alpha=0.5, label='Normal Orders', s=45
)
plt.scatter(
    flagged_orders['quantity'],
    flagged_orders['total_amount_kes'],
    c='#c8102e', alpha=0.9, label='Flagged Anomalies (Isolation Forest)', s=120, edgecolors='black', marker='X'
)

plt.title("Isolation Forest Anomaly Detection — Order Size vs. Total KES", fontsize=13, fontweight='bold')
plt.xlabel("Order Quantity (Cases/Units)")
plt.ylabel("Total Order Value (KSh)")
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()

print("\n=== Top 5 Flagged Anomalous Orders for Manager Review ===")
flagged_orders[['order_id', 'date', 'customer_name', 'product_name', 'quantity', 'total_amount_kes', 'anomaly_score']].head()

--- 
## 📋 Final Demonstration Summary for Examiners

| Module | Algorithm | Applied Dataset | Production Output & Integration |
| :--- | :--- | :--- | :--- |
| **1. Demand Forecasting** | SARIMAX $(1,1,1)(1,1,0)_7$ | 12-Month Daily Volumes (Litres) | Plotted as 30-day projected curve on Django Operations Dashboard |
| **2. Customer Segmentation** | K-Means Clustering ($k=3$) | Recency, Frequency, Monetary Matrix | Badged into customer accounts (VIP, Regular, At-Risk) in Django CRM |
| **3. Anomaly Detection** | Isolation Forest (5% Contam.) | Multi-variate Order Features | Real-time security alerts flagged in the Order Management Queue |